# Day 09. Exercise 02
# Metrics

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import joblib


## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df_features = pd.read_csv('../../src/data/day-of-week-not-scaled.csv')


In [3]:
df_target = pd.read_csv('../../src/data/dayofweek.csv')
y = df_target['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(
    df_features, y, test_size=0.2, random_state=21, stratify=y
)


## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [4]:
svm = SVC(C=10, gamma='auto', kernel='rbf', random_state=21, probability=True)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
y_prob = svm.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')

print(f'accuracy is {accuracy:.5f}')
print(f'precision is {precision:.5f}')
print(f'recall is {recall:.5f}')
print(f'roc_auc is {roc_auc:.5f}')


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.98168


## 3. Decision tree

1. The same task for decision tree

In [5]:
dt = DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=23, random_state=21)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)
y_prob = dt.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')

print(f'accuracy is {accuracy:.5f}')
print(f'precision is {precision:.5f}')
print(f'recall is {recall:.5f}')
print(f'roc_auc is {roc_auc:.5f}')


accuracy is 0.89349
precision is 0.89665
recall is 0.89349
roc_auc is 0.93795


## 4. Random forest

1. The same task for random forest.

In [6]:
rf = RandomForestClassifier(n_estimators=50, max_depth=28, random_state=21)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')

print(f'accuracy is {accuracy:.5f}')
print(f'precision is {precision:.5f}')
print(f'recall is {recall:.5f}')
print(f'roc_auc is {roc_auc:.5f}')


accuracy is 0.92899
precision is 0.93009
recall is 0.92899
roc_auc is 0.99151


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [7]:
best_model = rf
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
class_counts = y_test.value_counts().sort_index()

print('Errors by weekday:')
for i in class_counts.index:
    total = class_counts[i]
    errors = cm[i].sum() - cm[i, i]
    pct = (errors / total) * 100
    print(f'  Class {i}: {errors}/{total} = {pct:.2f}%')

worst_weekday = max(class_counts.index, key=lambda i: (cm[i].sum() - cm[i, i]) / class_counts[i] * 100)
print(f'\nMost errors for weekday {worst_weekday}')

errors_mask = y_pred != y_test
X_errors = X_test[errors_mask.values]
y_error_actual = y_test[errors_mask.values]

print('\nErrors analysis by labname:')
lab_cols = [c for c in df_features.columns if c.startswith('labname')]
for col in lab_cols:
    error_count = X_errors[col].sum()
    total = X_test[col].sum()
    if total > 0:
        pct = (error_count / total) * 100
        print(f'  {col}: {error_count}/{total} = {pct:.2f}%')

worst_lab = max(lab_cols, key=lambda c: X_errors[c].sum() / max(X_test[c].sum(), 1) * 100 if X_test[c].sum() > 0 else 0)
print(f'\nMost errors for labname {worst_lab}')

user_cols = [c for c in df_features.columns if c.startswith('uid')]
print('\nErrors analysis by user:')
for col in user_cols:
    error_count = X_errors[col].sum()
    total = X_test[col].sum()
    if total > 0:
        pct = (error_count / total) * 100
        print(f'  {col}: {error_count}/{total} = {pct:.2f}%')

worst_user = max(user_cols, key=lambda c: X_errors[c].sum() / max(X_test[c].sum(), 1) * 100 if X_test[c].sum() > 0 else 0)
print(f'\nMost errors for user {worst_user}')


Errors by weekday:
  Class 0: 7/27 = 25.93%
  Class 1: 6/55 = 10.91%
  Class 2: 2/30 = 6.67%
  Class 3: 2/80 = 2.50%
  Class 4: 3/21 = 14.29%
  Class 5: 3/54 = 5.56%
  Class 6: 1/71 = 1.41%

Most errors for weekday 0

Errors analysis by labname:
  labname_code_rvw: 1.0/13.0 = 7.69%
  labname_lab03: 1.0/1.0 = 100.00%
  labname_lab03s: 1.0/1.0 = 100.00%
  labname_lab05s: 1.0/6.0 = 16.67%
  labname_laba04: 6.0/35.0 = 17.14%
  labname_laba04s: 0.0/25.0 = 0.00%
  labname_laba05: 1.0/47.0 = 2.13%
  labname_laba06: 2.0/9.0 = 22.22%
  labname_laba06s: 2.0/15.0 = 13.33%
  labname_project1: 9.0/186.0 = 4.84%

Most errors for labname labname_lab03

Errors analysis by user:
  uid_user_1: 0.0/9.0 = 0.00%
  uid_user_10: 1.0/12.0 = 8.33%
  uid_user_12: 0.0/12.0 = 0.00%
  uid_user_13: 0.0/17.0 = 0.00%
  uid_user_14: 1.0/31.0 = 3.23%
  uid_user_15: 0.0/2.0 = 0.00%
  uid_user_16: 2.0/5.0 = 40.00%
  uid_user_17: 0.0/7.0 = 0.00%
  uid_user_18: 1.0/6.0 = 16.67%
  uid_user_19: 4.0/19.0 = 21.05%
  uid_user_2

In [8]:
joblib.dump(best_model, 'best_model_ex02.joblib')


['best_model_ex02.joblib']

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [9]:
def evaluate_models(models, params_list, X_train, y_train, X_test, y_test):
    results = {}
    for model_class, params in zip(models, params_list):
        model = model_class(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')

        results[model_class.__name__] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'roc_auc': roc_auc
        }
    return results

models = [SVC, DecisionTreeClassifier, RandomForestClassifier]
params_list = [
    {'C': 10, 'gamma': 'auto', 'kernel': 'rbf', 'random_state': 21, 'probability': True},
    {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 23, 'random_state': 21},
    {'n_estimators': 50, 'max_depth': 28, 'random_state': 21}
]

results = evaluate_models(models, params_list, X_train, y_train, X_test, y_test)
for model_name, metrics in results.items():
    print(f'\n{model_name}:')
    for metric, value in metrics.items():
        print(f'  {metric} = {value:.5f}')


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



SVC:
  accuracy = 0.88757
  precision = 0.89267
  recall = 0.88757
  roc_auc = 0.98168

DecisionTreeClassifier:
  accuracy = 0.89349
  precision = 0.89665
  recall = 0.89349
  roc_auc = 0.93795

RandomForestClassifier:
  accuracy = 0.92899
  precision = 0.93009
  recall = 0.92899
  roc_auc = 0.99151
